# 12. KNN Collaborative Filtering <a id="12-knn" name="12-knn"></a>

In [ ]:
# We simulate users rating movies because WILDFLIX does not have real users yet.
# Our solution is to simulate ratings using random values based on "vote_average".

np.random.seed(42)
N_USERS = 500
user_ids = [f"U{i:04d}" for i in range(1, N_USERS+1)]

# Sample movies for each user (simulate watch history)
movie_ids = df["id"].tolist()
user_movie_ratings = []
for uid in user_ids:
    n_watched = np.random.randint(10, 60)
    watched = np.random.choice(movie_ids, size=min(n_watched, len(movie_ids)), replace=False)
    for mid in watched:
        movie_row = df[df["id"] == mid]
        if len(movie_row):
            base = float(movie_row["vote_average"].values[0])
            simulated_rating = float(np.clip(np.random.normal(base, 0.8), 1.0, 10.0))
            user_movie_ratings.append({
                "user_id": uid, "movie_id": mid, "rating": round(simulated_rating, 1)
            })

df_ratings = pd.DataFrame(user_movie_ratings)

# User-item matrix
user_item = df_ratings.pivot_table(index="user_id", columns="movie_id", values="rating", fill_value=0)
print(f" User-Item matrix: {user_item.shape}")
print(f"   Users:     {user_item.shape[0]:,}")
print(f"   Movies:    {user_item.shape[1]:,}")
print(f"   Sparsity:  {1 - df_ratings.shape[0]/(user_item.shape[0]*user_item.shape[1]):.3%}")

# KNN on items
user_item_sparse = csr_matrix(user_item.values)
knn_model = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=20)
knn_model.fit(user_item_sparse.T)

mid_to_idx = {mid: i for i, mid in enumerate(user_item.columns)}
idx_to_mid = {i: mid for mid, i in mid_to_idx.items()}
print(f"\n KNN model trained (cosine similarity, item-based)")

 User-Item matrix: (500, 5048)
   Users:     500
   Movies:    5,048
   Sparsity:  99.330%

 KNN model trained (cosine similarity, item-based)


In [ ]:
def knn_recommendations(movie_title, n=5):
    """KNN item-based collaborative recommendations."""
    matched = fuzzy_match(movie_title, title_to_idx)
    if not matched: return None
    idx = title_to_idx[matched]
    if isinstance(idx, pd.Series): idx = idx.iloc[0]
    movie_id = df.iloc[idx]["id"]

    if movie_id not in mid_to_idx:
        print(f"  Movie not in rating matrix — using content-based fallback")
        return content_recommendations(movie_title, n=n)

    kidx = mid_to_idx[movie_id]
    dists, indices = knn_model.kneighbors(user_item_sparse.T[kidx], n_neighbors=n+1)

    neighbor_ids = [idx_to_mid[i] for i in indices.flatten() if i != kidx][:n]
    neighbor_sims = 1 - dists.flatten()[1:n+1]

    results = df[df["id"].isin(neighbor_ids)][
        ["id", "title", "release_year", "primary_genre", "vote_average", "runtime", "director"]
    ].copy()

    sim_map = dict(zip(neighbor_ids, neighbor_sims))
    results["knn_similarity"] = results["id"].map(sim_map)

    return results.drop(columns=["id"]).sort_values("knn_similarity", ascending=False).reset_index(drop=True)

print(" KNN Recommendations Demo:")
for movie in ["Star Wars", "Forrest Gump", "The Dark Knight"]:
    print(f"\n Input: {movie}")
    res = knn_recommendations(movie, n=5)
    if res is not None and not isinstance(res, tuple):
        print(res[["title", "primary_genre", "vote_average", "knn_similarity"]].to_string())

 KNN Recommendations Demo:

 Input: Star Wars
                      title primary_genre  vote_average  knn_similarity
0       Clash of the Titans     Adventure          6.90            0.51
1                    Tetris      Thriller          7.63            0.43
2   The Long Kiss Goodnight         Crime          6.60            0.43
3         Prayers for Bobby         Drama          8.06            0.43
4  Turtles All the Way Down         Drama          7.30            0.41

 Input: Forrest Gump
                               title primary_genre  vote_average  knn_similarity
0                           Fearless         Drama          7.51            0.55
1    Goosebumps 2: Haunted Halloween        Comedy          6.10            0.55
2                            Watcher        Horror          6.48            0.55
3  The Girl Who Believes in Miracles        Family          7.40            0.50
4                          Peter Pan     Adventure          7.14            0.48

 Input: The D